# 01 · Data cleaning

**Input:** `dataset/Reviews.csv`. **Output:** `outputs/data/reviews_clean.pkl`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

## 1. Problem summary

**Context.** Online shops receive thousands of product reviews every day. Reading them by hand is not feasible, yet the sentiment they carry shows a seller which products disappoint buyers and which are praised. This project builds a classifier that reads the text of a food review and predicts its sentiment.

**Task.** Three-class sentiment classification.
- **Input:** the title (`Summary`) and the body (`Text`) of an English review.
- **Output:** one label: *negative*, *neutral* or *positive*.

**Data.** [Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews) (McAuley & Leskovec, WWW 2013): 568,454 reviews posted between October 1999 and October 2012, each with a 1-5 star rating. The dataset has no human sentiment annotation.

**Labels (weak).** The star rating is mapped to a label once and for all:

| Stars | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Label | negative | negative | neutral | positive | positive |

**Why it is hard.**
- *Skewed labels:* about 78% of the cleaned reviews are positive and only 7.6% are neutral.
- *Ambiguous neutral class:* a 3-star review often praises and complains at once.
- *Weak labels:* a 5-star review can still contain a complaint.
- *Context:* negation ("not bad"), contrast ("good but...") and sarcasm need word order, not just word counts.
- *Dirty data:* about 31% of the raw rows are repeated reviews, plus HTML tags and very uneven lengths.

**Metric.** Macro-F1, the unweighted mean of the three class F1 scores: always answering *positive* already gives 78% accuracy but only 0.29 Macro-F1.

**Approach.** Clean the data and remove repeated reviews before splitting (notebooks 01-03), then compare two neural classifiers trained on the same split with a class-weighted loss: a **BiLSTM with attention** trained from scratch (04) and a **BERT-base** model fine-tuned on the reviews (05). The rule "always answer positive" is the reference. Notebook 06 evaluates and compares the models and 07 is an interactive demo.

| Notebook | Content |
|---|---|
| 01 data cleaning | problem summary, quality checks, removal of repeated reviews |
| 02 data understanding | exploratory analysis and the design decisions it implies |
| 03 data preparation | model input text, stratified split, class weights |
| 04 BiLSTM + attention | model 1, trained from scratch |
| 05 BERT-base | model 2, fine-tuned |
| 06 evaluation | results, bootstrap intervals, threshold calibration, error analysis |
| 07 demo | predictions for your own sentences |

## 2. Configuration

In [ ]:
import time
import pandas as pd
from IPython.display import display

from src.config import CFG, DATA_PATH, LABELS, SCORE_TO_LABEL, setup_notebook
from src.data import CLEAN_PATH

setup_notebook()
pd.Series(CFG, name='value').to_frame()

## 3. Load the data

In [ ]:
t0 = time.time()
raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head(1)

In [ ]:
raw.info()

## 4. Data quality checks

In [ ]:
DUP_KEY = ['UserId', 'ProfileName', 'Time', 'Text']

print('Missing values per column:')
print(raw.isna().sum())
print()
print('Fully duplicated rows     :', raw.duplicated().sum())
print('Repeated reviews (user, time, text):', raw.duplicated(subset=DUP_KEY).sum())
print('Repeated Text             :', raw['Text'].duplicated().sum())
print('Invalid helpfulness       :', (raw['HelpfulnessNumerator'] > raw['HelpfulnessDenominator']).sum())

In [ ]:
dup_groups = raw[raw.duplicated(subset=DUP_KEY, keep=False)]
example_key = dup_groups.iloc[0][DUP_KEY]
raw[(raw[DUP_KEY] == example_key).all(axis=1)][['Id', 'ProductId', 'UserId', 'Score', 'Time', 'Summary']]

## 5. Cleaning

In [ ]:
steps = [('Raw data', len(raw))]
df = raw.drop_duplicates(subset=DUP_KEY, keep='first')
steps.append(('Drop repeated reviews (user, time, text)', len(df)))
df = df[df['HelpfulnessNumerator'] <= df['HelpfulnessDenominator']]
steps.append(('Drop invalid helpfulness', len(df)))
df = df.drop_duplicates(subset=['Text'], keep='first')
steps.append(('Drop repeated Text', len(df)))

df = df.copy()
df['Summary'] = df['Summary'].fillna('').astype(str)
df['label'] = df['Score'].map(SCORE_TO_LABEL)
df['sentiment'] = df['label'].map(dict(enumerate(LABELS)))
df['date'] = pd.to_datetime(df['Time'], unit='s')
df = df.reset_index(drop=True)

clean_steps = pd.DataFrame(steps, columns=['Step', 'Rows left'])
display(clean_steps)

## 6. Save the cleaned data

In [ ]:
df.to_pickle(CLEAN_PATH)
print(f'Saved {len(df):,} reviews to {CLEAN_PATH}')
df[['ProductId', 'UserId', 'Score', 'sentiment', 'date', 'Summary', 'Text']].head()